In [1]:
#!pip install pandas
#!pip install scikit-learn
#!pip install datasets
#!pip install transformers
#!pip install torch
#!pip install matplotlib
#!pip install seaborn
#!pip install accelerate>=0.26.0

In [2]:
import os
import ast
from ast import literal_eval

import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import BertTokenizer
from transformers import BertForSequenceClassification
from transformers import Trainer, TrainingArguments

import pickle

import torch
import torch.nn as nn

import numpy as np
from scipy.special import expit
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, confusion_matrix, multilabel_confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
model = "bert-base-multilingual-cased"
model_name = "mBERT"
data = "convdata"
lr_categories = 2e-5
lr_levels = 2e-5
lr_time = 2e-5
num_epochs = 3
batch_size = 16

In [4]:
def combine_df(input_dir):
    dfs = []

    for file in os.listdir(input_dir):
        if file.endswith(".csv"):
            df = pd.read_csv(os.path.join(input_dir, file))
            dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)

    return combined_df

In [5]:
def load_df(input_path):
    file = os.path.basename(input_path)
    if file.endswith(".csv"):
        df = pd.read_csv(input_path)

    return df

In [6]:
category_names = [
    "B1300 Energy level",
    "B140 Attention functions",
    "B152 Emotional functions",
    "B440 Respiration functions",
    "B455 Exercise tolerance functions",
    "B530 Weight maintenance functions",
    "D450 Walking",
    "D550 Eating",
    "D840-859 Work and employment",
    "B280 Sensations of pain",
    "B134 Sleep functions",
    "D760 Family relationships",
    "B164 Higher-level cognitive functions",
    "D465 Moving around using equipment",
    "D410 Changing basic body position",
    "B230 Hearing functions",
    "D240 Handling stress and other psychological demands",
    "None"
]

In [ ]:
# Create special tokens
special_tokens = [f"[{name}]" for name in category_names] + ["[LEVELS]", "[TEXT]", "[HISTORY]"]

In [8]:
time_names = [
    "past",
    "present",
    "future",
    "None"
]

### Load the data

In [9]:
input_path_train = "conv_input_data/train"
train_df = combine_df(input_path_train)

In [10]:
input_path_dev = "conv_input_data/dev"
dev_df = combine_df(input_path_dev)

In [11]:
# Drop rows with NaN for text
train_df = train_df.dropna(subset=["text"])
dev_df = dev_df.dropna(subset=["text"])

## Categories

In [12]:
categories_train_df = train_df.copy()
categories_dev_df = dev_df.copy()

In [13]:
categories_train_df["categories"] = categories_train_df["categories"].apply(ast.literal_eval)
categories_dev_df["categories"] = categories_dev_df["categories"].apply(ast.literal_eval)

### Encode categories

In [14]:
invalid_rows = categories_dev_df[categories_dev_df["categories"].apply(lambda cats: any(cat not in category_names for cat in cats))]
print(f"There are {len(invalid_rows)} invalid rows in the development set.")

categories_train_df = categories_train_df.copy()
categories_train_df = categories_train_df[~categories_train_df["categories"].apply(
    lambda cats: any(cat not in category_names for cat in cats))]
categories_dev_df = categories_dev_df.copy()
categories_dev_df = categories_dev_df[~categories_dev_df["categories"].apply(
    lambda cats: any(cat not in category_names for cat in cats))]

There are 1562 invalid rows in the development set.


In [15]:
categories_mlb = MultiLabelBinarizer(classes=category_names)

categories_train_df["labels"] = list(categories_mlb.fit_transform(categories_train_df["categories"]))
categories_train_df["labels"] = categories_train_df["labels"].apply(lambda x: [float(v) for v in x])
categories_dev_df["labels"] = list(categories_mlb.transform(categories_dev_df["categories"]))
categories_dev_df["labels"] = categories_dev_df["labels"].apply(lambda x: [float(v) for v in x])

In [16]:
categories_train_dataset = Dataset.from_pandas(categories_train_df)
categories_dev_dataset = Dataset.from_pandas(categories_dev_df)

### Tokenise text with categories

In [17]:
# Load mBERT tokeniser
#categories_tokeniser = BertTokenizer.from_pretrained(model)
categories_tokeniser = BertTokenizer.from_pretrained(f"./{data}/{model_name}_categories")

In [18]:
#categories_tokeniser.save_pretrained(f"./{data}/{model_name}_categories")

In [19]:
def tokenise_categories_function(examples):
    return categories_tokeniser(examples["text"], 
                     truncation=True, 
                     padding="max_length", 
                     max_length=512)

In [20]:
"""categories_train_dataset = categories_train_dataset.map(tokenise_categories_function, batched=True)
categories_dev_dataset = categories_dev_dataset.map(tokenise_categories_function, batched=True)"""

'categories_train_dataset = categories_train_dataset.map(tokenise_categories_function, batched=True)\ncategories_dev_dataset = categories_dev_dataset.map(tokenise_categories_function, batched=True)'

In [21]:
"""categories_train_dataset.set_format(
    type="torch", 
    columns=["input_ids", "attention_mask", "labels"]
)

categories_dev_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)"""

'categories_train_dataset.set_format(\n    type="torch", \n    columns=["input_ids", "attention_mask", "labels"]\n)\n\ncategories_dev_dataset.set_format(\n    type="torch",\n    columns=["input_ids", "attention_mask", "labels"]\n)'

### Train the categories model

In [22]:
def categories_compute_metrics(eval_pred):
    logits, labels = eval_pred

    probabilities = torch.sigmoid(torch.tensor(logits))
    predictions = (probabilities > 0.5).int().numpy()

    return {"micro_f1": f1_score(labels, predictions, average="micro"),
           "macro_f1": f1_score(labels, predictions, average="macro"),
           "weighted_f1": f1_score(labels, predictions, average="weighted")}

In [23]:
"""categories_model = BertForSequenceClassification.from_pretrained(model, 
                                                                 num_labels=18,
                                                                 problem_type="multi_label_classification")"""
categories_model = BertForSequenceClassification.from_pretrained(f"./{data}/{model_name}_categories")
categories_trainer = Trainer(model=categories_model)

In [24]:
"""categories_training_args = TrainingArguments(
    output_dir=f"./{data}/results_categories",
    learning_rate=lr_categories,
    num_train_epochs=num_epochs,                    
    per_device_train_batch_size=batch_size,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir=f"./{data}/logs_categories",
)
 
categories_trainer = Trainer(
    model=categories_model,
    args=categories_training_args,
    train_dataset=categories_train_dataset,
    eval_dataset=categories_dev_dataset,               
    compute_metrics=categories_compute_metrics
)

categories_trainer.train()"""

'categories_training_args = TrainingArguments(\n    output_dir=f"./{data}/results_categories",\n    learning_rate=lr_categories,\n    num_train_epochs=num_epochs,                    \n    per_device_train_batch_size=batch_size,\n    eval_strategy="epoch",\n    save_strategy="epoch",\n    logging_dir=f"./{data}/logs_categories",\n)\n \ncategories_trainer = Trainer(\n    model=categories_model,\n    args=categories_training_args,\n    train_dataset=categories_train_dataset,\n    eval_dataset=categories_dev_dataset,               \n    compute_metrics=categories_compute_metrics\n)\n\ncategories_trainer.train()'

### Saving the categories model

In [25]:
# Save the trained categories model
#categories_model.save_pretrained(f"./{data}/{model_name}_categories")

## Levels

### Load the levels model

In [26]:
levels_train_df = train_df.copy()
levels_dev_df = dev_df.copy()

In [27]:
levels_train_df["categories"] = levels_train_df["categories"].apply(ast.literal_eval)
levels_dev_df["categories"] = levels_dev_df["categories"].apply(ast.literal_eval)
levels_train_df["levels"] = levels_train_df["levels"].apply(ast.literal_eval)
levels_dev_df["levels"] = levels_dev_df["levels"].apply(ast.literal_eval)

In [28]:
def duplicate_rows(df):
    expanded_rows = []
    
    for i, row in df.iterrows():
        categories = row["categories"]
        levels = row["levels"] 

        if len(categories) != len(levels):
            print(f"Skipping {row['turn_id']} from {row['conversation_id']}")
            continue
    
        if len(categories) == 0:
            categories = [None]

        for j, (cat, level) in enumerate(zip(categories, levels)):
            if level == "0":
                continue
            new_row = row.copy()
            new_row["full_turn_id"] = (f"{row['conversation_id']}_{row['turn_id']}")
            new_row["new_turn_id"] = f"{row['turn_id']}_{j}"
            new_row["category"] = str(cat)
            new_row["level"] = level

            expanded_rows.append(new_row)
    
    return pd.DataFrame(expanded_rows)

levels_train_df = duplicate_rows(levels_train_df)
levels_dev_df = duplicate_rows(levels_dev_df)

### Augment data 

In [29]:
levels_train_df["levels"] = levels_train_df["levels"].astype(str)
levels_dev_df["levels"] = levels_dev_df["levels"].astype(str)

In [30]:
levels_train_df["category"] = levels_train_df["category"].fillna("None")
levels_dev_df["category"] = levels_dev_df["category"].fillna("None")
levels_train_df["level"] = levels_train_df["level"].fillna("None")
levels_dev_df["level"] = levels_dev_df["level"].fillna("None")

In [31]:
def build_levels_input(df, history_size=5):
    turn_categories = {}
    for (conv_id, turn_id), turn_rows in df.groupby(["conversation_id", "full_turn_id"], sort=False):
        turn_categories[(conv_id, turn_id)] = "".join(f"[{cat}]" for cat in turn_rows["category"])

    turn_histories  = {}

    for conv_id, conv_df in df.groupby("conversation_id", sort=False):
        turn_ids = conv_df["full_turn_id"].drop_duplicates().tolist()

        for i, turn_id in enumerate(turn_ids):
            previous_turns = turn_ids[max(0, i-history_size):i]

            history = " ".join(turn_categories[(conv_id, prev_turn)] for prev_turn in previous_turns)

            turn_histories[(conv_id, turn_id)] = history

    return df.apply(lambda row:
                    f"[HISTORY] {turn_histories[(row['conversation_id'], row['full_turn_id'])]} "
                    f"[CURRENT] [{row['category']}] "
                    f"[TEXT] {row['text']}",
            axis=1)


In [32]:
# Combine the predicted categories with the original text for the level model input
#levels_train_df["combined_text"] = build_levels_input(levels_train_df)
#levels_dev_df["combined_text"] = build_levels_input(levels_dev_df)
levels_train_df = pd.read_csv("./results_intermission/level_train_df.csv")
levels_dev_df = pd.read_csv("./results_intermission/level_dev_df.csv")

In [33]:
#levels_train_df.to_csv(f"./results_intermission/level_train_df.csv")
#levels_dev_df.to_csv(f"./results_intermission/level_dev_df.csv")

### Encode levels

In [34]:
levels_train_df["labels"] = levels_train_df["level"].apply(lambda x: -1 if pd.isna(x) or x == "None" or x is None or x == "" else float(x))
levels_dev_df["labels"] = levels_dev_df["level"].apply(lambda x: -1 if pd.isna(x) or x == "None" or x is None or x == "" else float(x))

In [35]:
levels_train_df["level"] = levels_train_df["level"].astype(str)
levels_dev_df["level"] = levels_dev_df["level"].astype(str)

In [36]:
levels_train_dataset = Dataset.from_pandas(levels_train_df)
levels_dev_dataset = Dataset.from_pandas(levels_dev_df)

### Tokenise text with levels

In [37]:
# Load mBERT tokeniser
levels_tokeniser = BertTokenizer.from_pretrained(model)
#levels_tokeniser = BertTokenizer.from_pretrained(f"./{data}/{model_name}_levels")

In [38]:
levels_tokeniser.add_special_tokens(
    {"additional_special_tokens": special_tokens}
)

21

In [ ]:
levels_tokeniser.save_pretrained(f"./{data}/{model_name}_levels")

('./convdata/mBERT_levels_no0/tokenizer_config.json',
 './convdata/mBERT_levels_no0/special_tokens_map.json',
 './convdata/mBERT_levels_no0/vocab.txt',
 './convdata/mBERT_levels_no0/added_tokens.json')

In [ ]:
def tokenise_levels_function(examples):
    return levels_tokeniser(examples["combined_text"], 
                     truncation=True, 
                     padding="max_length", 
                     max_length=512)

In [41]:
levels_train_dataset = levels_train_dataset.map(tokenise_levels_function, batched=True)
levels_dev_dataset = levels_dev_dataset.map(tokenise_levels_function, batched=True)

Map:   0%|          | 0/400026 [00:00<?, ? examples/s]

Map:   0%|          | 0/105972 [00:00<?, ? examples/s]

In [42]:
levels_train_dataset.set_format(
    type="torch", 
    columns=["input_ids", "attention_mask", "labels"]
)

levels_dev_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

### Train the levels model

In [43]:
class MaskedRegressionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits.squeeze()

        mask = labels != -1

        loss_fct = nn.MSELoss()

        loss = loss_fct(
                logits[mask],
                labels.float()[mask])

        return (loss, outputs) if return_outputs else loss

In [44]:
def levels_compute_metrics(eval_pred):
    preds, labels = eval_pred
    predictions = preds.squeeze()

    mask = (labels != -1) & ~np.isnan(labels)
    labels_filtered = labels[mask]
    predictions_filtered = predictions[mask]

    mae = mean_absolute_error(labels, predictions)
    mse = mean_squared_error(labels, predictions)
    rmse = root_mean_squared_error(labels, predictions)

    return {"mae": mae,
            "mse": mse,
            "rmse": rmse
            }

In [45]:
levels_model = BertForSequenceClassification.from_pretrained(model, 
                                                              num_labels=1, #len(level_encoder.classes_),
                                                              problem_type="regression")
#levels_model = BertForSequenceClassification.from_pretrained(f"./{data}/{model_name}_levels")
#levels_trainer = Trainer(model=levels_model)
levels_model.resize_token_embeddings(len(levels_tokeniser))

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 04b5c058-1b06-490d-844e-f7d2846d4d26)')' thrown while requesting HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/config.json
Retrying in 1s [Retry 1/5].
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(119568, 768, padding_idx=0)

In [46]:
levels_training_args = TrainingArguments(
    output_dir=f"./{data}/results_levels",
    learning_rate=lr_levels,
    num_train_epochs=num_epochs,                    
    per_device_train_batch_size=batch_size,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir=f"./{data}/logs_levels",
)

levels_trainer = MaskedRegressionTrainer(
    model=levels_model,
    args=levels_training_args,
    train_dataset=levels_train_dataset,
    eval_dataset=levels_dev_dataset,   
    tokenizer=levels_tokeniser,
    compute_metrics=levels_compute_metrics
)

levels_trainer.train()

/tmp/P103572.1970859/ipykernel_2801117/1200764627.py:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `MaskedRegressionTrainer.__init__`. Use `processing_class` instead.
  levels_trainer = MaskedRegressionTrainer(


Epoch,Training Loss,Validation Loss,Mae,Mse,Rmse
1,2.510700,nan,1.791263,3.553015,1.884944
2,2.878200,nan,1.996499,4.301533,2.074014
3,2.411100,nan,1.881366,3.869931,1.967214


TrainOutput(global_step=75006, training_loss=2.59887063150752, metrics={'train_runtime': 12071.2462, 'train_samples_per_second': 99.416, 'train_steps_per_second': 6.214, 'total_flos': 3.157509540684534e+17, 'train_loss': 2.59887063150752, 'epoch': 3.0})

### Saving the levels model

In [ ]:
# Save the trained levels model
levels_model.save_pretrained(f"./{data}/{model_name}_levels")

## Time

In [48]:
time_train_df = train_df.copy()
time_dev_df = dev_df.copy()

In [49]:
#time_train_df["categories"] = time_train_df["categories"].apply(ast.literal_eval)
#time_dev_df["categories"] = time_dev_df["categories"].apply(ast.literal_eval)
#time_train_df["levels"] = time_train_df["levels"].apply(ast.literal_eval)
#time_dev_df["levels"] = time_dev_df["levels"].apply(ast.literal_eval)
time_train_df["relative_time"] = time_train_df["relative_time"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else ["None"])
time_dev_df["relative_time"] = time_dev_df["relative_time"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else ["None"])

### Augment data 

In [50]:
time_train_df["levels"] = time_train_df["levels"].fillna("None")
time_dev_df["levels"] = time_dev_df["levels"].fillna("None")

In [51]:
time_train_df["categories"] = time_train_df["categories"].apply(ast.literal_eval)
time_dev_df["categories"] = time_dev_df["categories"].apply(ast.literal_eval)
time_train_df["levels"] = time_train_df["levels"].apply(ast.literal_eval)
time_dev_df["levels"] = time_dev_df["levels"].apply(ast.literal_eval)

In [52]:
def build_time_input(df, history_size=5):
    turn_pairs = []

    for row in df.itertuples():
        pairs = " ".join(
                f"[{cat}] {level}"
                for cat, level in zip(row.categories, row.levels)
            )
        turn_pairs.append(pairs)

    df = df.copy()
    df["pair_text"] = turn_pairs

    histories = []

    for conv_id, conv_df in df.groupby("conversation_id", sort=False):
        history = []

        for row in conv_df.itertuples():
            histories.append(" [SEP] ".join(history[-history_size:]))
            history.append(row.pair_text)

    df["history"] = histories

    return ("[HISTORY] " + df["history"]
            + " [CURRENT] " + df["pair_text"]
            + " [TEXT] " + df["text"].astype(str))

In [53]:
# Combine the predicted categories with teh original text for the level model input
time_train_df["double_combined_text"] = build_time_input(time_train_df)
time_dev_df["double_combined_text"] = build_time_input(time_dev_df)
#time_train_df = pd.read_csv("./results_intermission/time_train_df.csv")
#time_dev_df = pd.read_csv("./results_intermission/time_dev_df.csv")

In [54]:
time_train_df.to_csv(f"./results_intermission/time_train_df.csv")
time_dev_df.to_csv(f"./results_intermission/time_dev_df.csv")

In [55]:
def has_no_labels(row):
    return (all(cat == ["None"] for cat in row["categories"])
                and all(level == ["None"] for level in row["levels"]))

In [56]:
# Remove rows from training data if they contain no levels or categories
time_train_df = time_train_df[~time_train_df.apply(has_no_labels, axis=1)].copy()
time_dev_df = time_dev_df[~time_dev_df.apply(has_no_labels, axis=1)].copy()

### Encode time

In [57]:
time_mlb = MultiLabelBinarizer(
    classes=time_names
)

time_train_df["labels"] = list(time_mlb.fit_transform(time_train_df["relative_time"]))
time_train_df["labels"] = time_train_df["labels"].apply(lambda x: [float(v) for v in x])
time_dev_df["labels"] = list(time_mlb.transform(time_dev_df["relative_time"]))
time_dev_df["labels"] = time_dev_df["labels"].apply(lambda x: [float(v) for v in x])

In [58]:
time_train_dataset = Dataset.from_pandas(time_train_df[["double_combined_text", "labels"]])
time_dev_dataset = Dataset.from_pandas(time_dev_df[["double_combined_text", "labels"]])

### Tokenise text with time

In [59]:
# Load mBERT tokeniser
#time_tokeniser = BertTokenizer.from_pretrained(model)
time_tokeniser = BertTokenizer.from_pretrained(f"./{data}/{model_name}_time")

In [60]:
time_tokeniser.add_special_tokens(
    {"additional_special_tokens": special_tokens}
)

0

In [61]:
#time_tokeniser.save_pretrained(f"./{data}/{model_name}_time_no0")

In [62]:
def tokenise_time_function(examples):
    return time_tokeniser(examples["double_combined_text"], 
                     truncation=True,  
                     padding="max_length", 
                     max_length=512)

In [63]:
"""time_train_dataset = time_train_dataset.map(tokenise_time_function, batched=True)
time_dev_dataset = time_dev_dataset.map(tokenise_time_function, batched=True)"""

'time_train_dataset = time_train_dataset.map(tokenise_time_function, batched=True)\ntime_dev_dataset = time_dev_dataset.map(tokenise_time_function, batched=True)'

In [64]:
"""time_train_dataset.set_format(
    type="torch", 
    columns=["input_ids", "attention_mask", "labels"]
)

time_dev_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)"""

'time_train_dataset.set_format(\n    type="torch", \n    columns=["input_ids", "attention_mask", "labels"]\n)\n\ntime_dev_dataset.set_format(\n    type="torch",\n    columns=["input_ids", "attention_mask", "labels"]\n)'

### Train the time model

In [65]:
def time_compute_metrics(eval_pred):
    logits, labels = eval_pred

    probabilities = expit(logits)
    predictions = (probabilities > 0.5).astype(int)

    predicted_times = time_mlb.inverse_transform(predictions)

    return {"accuracy": accuracy_score(labels, predictions),
            "macro_f1": f1_score(labels, predictions, average="macro"),
            "weighted_f1": f1_score(labels, predictions, average="weighted")}

In [66]:
"""time_model = BertForSequenceClassification.from_pretrained(model,
                                                           num_labels=len(time_names),
                                                           problem_type="multi_label_classification"
                                                           )"""
time_model = BertForSequenceClassification.from_pretrained(f"./{data}/{model_name}_time")
time_trainer = Trainer(model=time_model)
time_model.resize_token_embeddings(len(time_tokeniser))

Embedding(119568, 768, padding_idx=0)

In [67]:
"""time_training_args = TrainingArguments(
    output_dir=f"./{data}/results_time",
    learning_rate=lr_time,
    num_train_epochs=num_epochs,                    
    per_device_train_batch_size=batch_size,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir=f"./{data}/logs_time",
)

time_trainer = Trainer(
    model=time_model,
    args=time_training_args,
    train_dataset=time_train_dataset,
    eval_dataset=time_dev_dataset,                
    compute_metrics=time_compute_metrics
)

time_trainer.train()#resume_from_checkpoint=True)"""

'time_training_args = TrainingArguments(\n    output_dir=f"./{data}/results_time",\n    learning_rate=lr_time,\n    num_train_epochs=num_epochs,                    \n    per_device_train_batch_size=batch_size,\n    eval_strategy="epoch",\n    save_strategy="epoch",\n    logging_dir=f"./{data}/logs_time",\n)\n\ntime_trainer = Trainer(\n    model=time_model,\n    args=time_training_args,\n    train_dataset=time_train_dataset,\n    eval_dataset=time_dev_dataset,                \n    compute_metrics=time_compute_metrics\n)\n\ntime_trainer.train()#resume_from_checkpoint=True)'

### Saving the time model

In [68]:
# Save the trained time model
#time_model.save_pretrained(f"./{data}/{model_name}_time_no0")